# Fragrance Recommender met vectordb2

Een lichtgewicht alternatief voor de ChromaDB chatbot die we eerder bouwden. `vectordb2` is een minimale vector store van Kagi: een paar regels code voor werkende RAG search, en het gebruikt standaard `fastembed` wat sneller is en beter scoort op MTEB dan MiniLM.

## Beste aanpak voor queries zoals 'spicy perfume for a date'

Het resultaat van de recommender staat of valt met wat we embedden. Een goede `embedding_text` is een tekst-blob per parfum die zowel expliciete features bevat (notes, brand, gender) als impliciete vibes uit de Fragrantica `description`. Concreet:

1. **Een rijke tekst per parfum bouwen** — title + brand + gender + notes + description samenvoegen. De description bevat vaak woorden als *evening*, *seductive*, *fresh*, *daytime* die queries als "for a date" matchen zonder dat we ze expliciet hoeven te labelen.
2. **Embedden met fastembed (BGE-small)** — de default van vectordb2. Sneller dan sentence-transformers en hogere MTEB-score dan all-MiniLM-L6-v2.
3. **Metadata apart opslaan** — rating, gender, brand, url — zodat we na retrieval kunnen filteren en sorteren. vectordb2 heeft geen WHERE-clause zoals ChromaDB.
4. **Re-ranken op rating** — bij gelijke similarity wint het hoger gerate parfum. Dat is wat een mens ook zou doen.
5. **Query expansion (optioneel)** — "spicy for date": voeg gangbare notes toe ("cinnamon, pepper, saffron, oud, amber, warm, evening") voor het embedden. Geeft betere recall bij korte queries.

## 0. Installatie

In [4]:
%pip install -q vectordb2 pandas spacy
!python -m spacy download en_core_web_sm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\alr3m\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
Traceback (most recent call last):
  File "<frozen runpy>", line 189, in _run_module_as_main
  File "<frozen runpy>", line 148, in _get_module_details
  File "<frozen runpy>", line 112, in _get_module_details
  File "C:\Users\alr3m\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\spacy\__init__.py", line 6, in <module>
    from .errors import setup_default_warnings
  File "C:\Users\alr3m\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\spacy\errors.py", line 3, in <module>
    from .compat import Literal
  File "C:\Users\alr3m\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\loc

**TODO:** De notes-kolom van Luckyscent bevat nog scraper-ruis die eruit moet. Bijvoorbeeld teksten zoals "Let us help you find your next signature scent!" die helemaal geen notes zijn maar marketingtekst van de pagina.

In [2]:
import pandas as pd
import ast
from pathlib import Path
from vectordb import Memory

pd.set_option('display.max_colwidth', 120)
print('Imports OK')

OSError: [WinError 4551] Zasady kontroli aplikacji zablokowały ten plik. Error loading "C:\Users\alr3m\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\lib\torch_python.dll" or one of its dependencies.

## 1. Data inladen en samenvoegen

We voegen drie bronnen samen tot één dataset:

1. **Fragrantica master** (`perfumes_table.csv`, 84k) — de originele dataset
2. **Filled-missing batches** (`fragnantica_missing_filled` + `missing_part1/2/3_filled`) — rijen die eerst leeg waren en die we aangevuld hebben via Parfumo. We matchen op `url` en overschrijven NaN-cellen in de master.
3. **Luckyscent** (`luckyscent_fragrances.csv`, 3k) — andere bron, ander schema. Heeft schone `gender` en `style` kolommen. We normaliseren naar gedeelde kolommen en concatten.

Daarna een kwaliteitsfilter: we houden alleen rijen met `reviews` en `notes`, want zonder notes valt er niks zinnigs te embedden. Reviews bevatten echte gebruikerstaal ("perfect for a date", "lasts all day") — dat is precies de vocabulaire die queries als *"spicy for a date"* matcht. Luckyscent valt hier af omdat die bron geen reviews heeft.

In [ ]:
# Fragrantica master + alle filled-missing batches 
master = pd.read_csv('fragrantica_data/perfumes_table.csv')

FILLED_FILES = [
    'fragrantica_data/fragnantica_missing_filled.csv',
    'fragrantica_data/missing_part1_filled.csv',
    'fragrantica_data/missing_part2_filled.csv',
    'fragrantica_data/missing_part3_filled.csv',
]
filled_parts = []
for fp in FILLED_FILES:
    part = pd.read_csv(fp)
    print(f'  {fp}: {len(part):,} rijen')
    filled_parts.append(part)
filled = pd.concat(filled_parts, ignore_index=True)
print(f'Filled-missing totaal (vóór dedupe): {len(filled):,}')

# Dedupe op url: behoud meest "ingevulde" rij per url (minste NaNs wint)
filled['_nonnull'] = filled.notna().sum(axis=1)
filled = (
    filled.sort_values('_nonnull', ascending=False)
          .drop_duplicates(subset='url', keep='first')
          .drop(columns='_nonnull')
)
print(f'Filled-missing na dedupe: {len(filled):,} unieke urls')

# Parfumo rating is 0-10, fragrantica is 0-5. normalized_rating is parfumo/10.
# Voor de filled rijen vervangen we 'rating' door normalized_rating × 5,
# zodat alles op 0-5 schaal staat.
if 'normalized_rating' in filled.columns:
    nr = pd.to_numeric(filled['normalized_rating'], errors='coerce')
    filled['rating'] = (nr * 5).where(nr.notna(), pd.to_numeric(filled['rating'], errors='coerce'))

# Drop kolommen die we niet meer nodig hebben vóór de merge
DROP_COLS = ['best_rating', 'rating_count', 'normalized_rating', 'parfumo_url']
filled = filled.drop(columns=[c for c in DROP_COLS if c in filled.columns])

# Merge in master (match op url) — update() raakt alleen bestaande kolommen
master = master.drop_duplicates(subset='url', keep='first').set_index('url')
filled_idx = filled.set_index('url')
master.update(filled_idx)
master = master.reset_index()

# --- Dedupe op (title, designer) — vangt Fragrantica's dubbele listings
# (zelfde parfum, twee URL-IDs zoals Gingham Love 77508 & 72692). Behoud de rij
# met de meeste non-null cellen (meestal degene mét reviews).
def _norm(s):
    return ' '.join(str(s).lower().split()) if pd.notna(s) else ''

master['_t'] = master['title'].apply(_norm)
master['_d'] = master['designer'].apply(_norm)
master['_nonnull'] = master.notna().sum(axis=1)

before_dedupe = len(master)
master = (
    master[master['_t'] != '']  # rijen zonder title kan je niet matchen
    .sort_values('_nonnull', ascending=False)
    .drop_duplicates(subset=['_t', '_d'], keep='first')
    .drop(columns=['_t', '_d', '_nonnull'])
    .reset_index(drop=True)
)
print(f'(Title, designer)-dedupe: {before_dedupe:,} → {len(master):,}  (-{before_dedupe-len(master)})')
print(f'\nFragrantica (incl. filled, gededupliceerd): {len(master):,} parfums')

# Luckyscent normaliseren naar gedeelde schema
ls = pd.read_csv('luckyscent_data/luckyscent_fragrances.csv')
ls_norm = pd.DataFrame({
    'url':         ls['url'],
    'title':       ls['name'].fillna('') + ' — ' + ls['brand'].fillna(''),
    'designer':    ls['brand'],
    'notes':       ls['notes'],
    'description': ls['description'],
    'rating':      pd.NA,
    'reviews':     pd.NA,
    'gender_raw':  ls['gender'],
    'style':       ls['style'],
})
print(f'Luckyscent: {len(ls_norm):,} parfums')

# Concat 
df = pd.concat([master, ls_norm], ignore_index=True)
print(f'\nTotaal gecombineerd: {len(df):,} parfums')

#  Kwaliteitsfilter: alleen rijen met gevulde reviews + notes 
# Reviews zijn de belangrijkste signal voor vibe-matching ("for a date" etc).
# Notes blijven nodig omdat ze in de embedding_text gaan.
def _is_nonempty(raw):
    if pd.isna(raw):
        return False
    s = str(raw).strip()
    return s not in ('', '[]', 'nan', 'None')

mask = df['reviews'].apply(_is_nonempty) & df['notes'].apply(_is_nonempty)

before = len(df)
df = df[mask].reset_index(drop=True)

df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

print(f'\nKwaliteitsfilter (reviews + notes): {before:,} → {len(df):,}')
print(f"Rating range na merge: {df['rating'].min():.2f} – {df['rating'].max():.2f}  (mean {df['rating'].mean():.2f})")
df.head(2)

## 2. Embedding-tekst bouwen

Als de pipeline al een `embedding_text` kolom heeft gebruiken we die direct. Anders bouwen we hem hier: **title + brand + gender + notes + description** samenvoegen. Dit is de kerntekst die per parfum geembed wordt.

In [ ]:
import re, html
from collections import defaultdict


# noise stripping 
_URL_RE       = re.compile(r'https?://\S+')
_SHOW_MORE_RE = re.compile(r'\b(?:show|read)\s+more\b', re.I)
_REPEAT_PUNCT = re.compile(r'([!?.,])\1{2,}')
_WHITESPACE   = re.compile(r'\s+')

def clean_review(text):
    """Strip HTML entities, URLs, 'show more' artifacts, repeated punctuation,
    collapse whitespace."""
    if not text:
        return ''
    s = html.unescape(str(text))
    s = _URL_RE.sub(' ', s)
    s = _SHOW_MORE_RE.sub(' ', s)
    s = _REPEAT_PUNCT.sub(r'\1\1', s)  # !!!!! -> !!
    s = _WHITESPACE.sub(' ', s).strip()
    return s


def _raw_review_list(raw):
    """Reviews-kolom -> Python list of raw strings."""
    if pd.isna(raw):
        return []
    s = str(raw).strip()
    if not s:
        return []
    if s.startswith('['):
        try:
            return [str(r) for r in ast.literal_eval(s)]
        except (ValueError, SyntaxError):
            return [s]
    return [s]


#  find reviews shared across multiple perfumes 
MIN_REVIEW_LEN = 50           # drop snippets shorter than this
DUP_THRESHOLD  = 3            # drop reviews that appear in >= N different perfumes

_review_rows = defaultdict(set)
for _i, _raw in df['reviews'].items():
    for _r in _raw_review_list(_raw):
        _cleaned = clean_review(_r).lower()
        if len(_cleaned) >= MIN_REVIEW_LEN:
            _review_rows[_cleaned].add(_i)

DUP_REVIEWS = {k for k, rows in _review_rows.items() if len(rows) >= DUP_THRESHOLD}
print(f'Cross-perfume duplicate review texts (>={DUP_THRESHOLD} perfumes): {len(DUP_REVIEWS):,}')
del _review_rows


# parsers 
def parse_notes(raw):
    """Notes-kolom is óf een string-van-Python-lijst (Fragrantica) óf comma-separated (Luckyscent)."""
    if pd.isna(raw) or not str(raw).strip():
        return ''
    s = str(raw).strip()
    if s.startswith('['):
        try:
            notes = ast.literal_eval(s)
            return ', '.join(str(n) for n in notes)
        except (ValueError, SyntaxError):
            pass
    return s


def parse_reviews(raw, max_reviews=5, max_chars=1500):
    """Clean + filter reviews. Drops: te kort, scraper-noise-only, en reviews
    die in >=DUP_THRESHOLD verschillende parfums voorkomen (scraper bleed-through).
    Sorteert op lengte aflopend zodat we de meest informatieve reviews houden
    binnen het char-budget."""
    items = _raw_review_list(raw)
    if not items:
        return ''

    candidates = []
    for r in items:
        cleaned = clean_review(r)
        if len(cleaned) < MIN_REVIEW_LEN:
            continue
        if cleaned.lower() in DUP_REVIEWS:
            continue
        candidates.append(cleaned)

    # Langste eerst (meeste semantische signaal per char-budget)
    candidates.sort(key=len, reverse=True)

    out, used = [], 0
    for r in candidates[:max_reviews]:
        if used + len(r) > max_chars:
            r = r[:max_chars - used]
        if not r:
            break
        out.append(r)
        used += len(r)
        if used >= max_chars:
            break
    return ' || '.join(out)


def infer_gender(row):
    """Luckyscent heeft schone gender-kolom; Fragrantica zet 'm in de title."""
    raw = row.get('gender_raw')
    if isinstance(raw, str) and raw.strip():
        g = raw.strip().lower()
        if g in ('men', 'women', 'unisex'):
            return g
    title = row.get('title', '')
    if not isinstance(title, str):
        return 'unisex'
    t = title.lower()
    if 'for women and men' in t or 'unisex' in t:
        return 'unisex'
    if 'for women' in t:
        return 'women'
    if 'for men' in t:
        return 'men'
    return 'unisex'


df['notes_text']   = df['notes'].apply(parse_notes)
df['reviews_text'] = df['reviews'].apply(parse_reviews)
df['gender']       = df.apply(infer_gender, axis=1)

# Style is alleen aanwezig voor Luckyscent — voeg toe als het er is
style_part = df.get('style', pd.Series([''] * len(df))).fillna('').astype(str)
style_part = style_part.where(style_part == '', 'Style: ' + style_part + '. ')

# "User reviews:" suffix valt weg als reviews_text leeg is — voorkomt
# dangling phrase die de embedder verwart.
user_reviews_part = df['reviews_text'].where(df['reviews_text'].str.len() == 0,
                                             'User reviews: ' + df['reviews_text'])

df['embedding_text'] = (
    df['title'].fillna('').astype(str) + '. '
    + 'Brand: ' + df['designer'].fillna('').astype(str) + '. '
    + 'Gender: ' + df['gender'] + '. '
    + style_part
    + 'Notes: ' + df['notes_text'] + '. '
    + df['description'].fillna('').astype(str) + ' '
    + user_reviews_part
)

df = df.dropna(subset=['embedding_text']).reset_index(drop=True)
df = df[df['embedding_text'].str.len() > 30].reset_index(drop=True)

# Rijen waarvan ALLE reviews gefilterd zijn behouden we — die hebben nog
# steeds signaal uit description + notes. Wel rapporteren hoeveel er zijn.
empty_reviews = (df['reviews_text'].str.len() == 0).sum()
print(f'\nKlaar voor embedding: {len(df):,} parfums')
print(f'Gem. reviews_text lengte: {df["reviews_text"].str.len().mean():.0f} chars')
print(f'Rijen waarvan alle reviews uitgefilterd zijn: {empty_reviews:,} '
      f'({empty_reviews/len(df)*100:.1f}%) — behouden vanwege description+notes')
print('\nVoorbeeld:')
print(df.iloc[0]['embedding_text'][:800])

In [ ]:
SEED = 42
TEST_FRAC = 0.20

shuffled = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
split_at = int(len(shuffled) * (1 - TEST_FRAC))
train_df = shuffled.iloc[:split_at].reset_index(drop=True)
test_df  = shuffled.iloc[split_at:].reset_index(drop=True)

print(f'Train: {len(train_df):,}  ({len(train_df)/len(df)*100:.1f}%)')
print(f'Test : {len(test_df):,}  ({len(test_df)/len(df)*100:.1f}%)')
print(f'Rating mean — train: {train_df["rating"].mean():.2f}  test: {test_df["rating"].mean():.2f}')

## 2b. Train/test split (80/20, seed=42)

Niet om overtraining tegen te gaan — er wordt hier niks getraind, de embeddings komen uit een frozen pretrained model. Wel om eerlijk te evalueren: als alle parfums in de index zitten kan een testquery een rij ophalen die we ook als ground truth gebruiken. Door 20% apart te houden houden we een schone testset over waarop we later queries als *"spicy perfume"* kunnen afzetten tegen de notes en reviews van de opgehaalde rijen.

# Index alleen op train_df — test_df houden we apart voor evaluatie
if len(memory.memory) == 0:
    texts = train_df['embedding_text'].tolist()

    meta_cols = ['title', 'designer', 'gender', 'rating', 'notes_text', 'reviews_text', 'url']
    if 'style' in train_df.columns:
        meta_cols.append('style')

    metadata = (
        train_df[meta_cols]
        .rename(columns={'designer': 'brand', 'notes_text': 'notes', 'reviews_text': 'reviews'})
        .astype(object)  # NaN → None bij to_dict
        .where(train_df[meta_cols].notna(), None)
        .to_dict('records')
    )

    memory.save(texts, metadata)
    print(f'Opgeslagen: {len(memory.memory):,} parfums (train set)')
else:
    print(f'Memory was al gevuld — {len(memory.memory):,} parfums geladen van disk.')

In [ ]:
MEMORY_FILE = 'vectordb_fragrances.pkl'

# Default embedder = fastembed BGE-small. chunking_strategy=sliding window 256.
memory = Memory(
    memory_file=MEMORY_FILE,
    chunking_strategy={'mode': 'sliding_window', 'window_size': 256, 'overlap': 0},
)

print(f'Memory init OK. Bestaande entries: {len(memory.memory)}')

TypeError: Memory.__init__() got an unexpected keyword argument 'embeddings'

In [ ]:
# Alleen vullen als-ie nog leeg is (eerste run kan even duren — ~5-15 min voor 87k)
if len(memory.memory) == 0:
    texts = df['embedding_text'].tolist()

    meta_cols = ['title', 'designer', 'gender', 'rating', 'notes_text', 'url']
    if 'style' in df.columns:
        meta_cols.append('style')

    metadata = (
        df[meta_cols]
        .rename(columns={'designer': 'brand', 'notes_text': 'notes'})
        .astype(object)  # NaN → None bij to_dict
        .where(df[meta_cols].notna(), None)
        .to_dict('records')
    )

    memory.save(texts, metadata)
    print(f'Opgeslagen: {len(memory.memory):,} parfums')
else:
    print(f'Memory was al gevuld — {len(memory.memory):,} parfums geladen van disk.')

In [ ]:
VIBE_HINTS = {
    'date':       'evening seductive sensual warm intimate',
    'office':     'clean professional subtle fresh daytime',
    'summer':     'fresh citrus light aquatic bright',
    'winter':     'warm amber spicy oud heavy cozy',
    'gym':        'fresh sporty clean light aquatic',
    'wedding':    'elegant sophisticated floral refined',
    'spicy':      'cinnamon pepper saffron cardamom clove ginger warm oriental',
    'sweet':      'vanilla caramel honey tonka gourmand',
    'woody':      'cedar sandalwood oud patchouli vetiver',
    'fresh':      'citrus bergamot lemon aquatic green',
    'floral':     'rose jasmine lily peony iris',
}

def expand_query(q: str) -> str:
    extras = [hint for keyword, hint in VIBE_HINTS.items() if keyword in q.lower()]
    return q if not extras else f"{q}. Related notes and vibe: {' '.join(extras)}"


def recommend(query: str, top_k: int = 5, gender: str | None = None,
              min_rating: float = 0.0, rating_weight: float = 0.2) -> pd.DataFrame:
    """
    query         : vrije tekst, bijv. 'spicy perfume for a date'
    gender        : 'men' | 'women' | 'unisex' | None
    min_rating    : hard filter op Fragrantica-rating
    rating_weight : 0 = puur similarity, 1 = puur rating. 0.2 is meestal lekker.
    """
    expanded = expand_query(query)
    # Haal meer kandidaten op dan we tonen → ruimte om te filteren & rerankien
    raw = memory.search(expanded, top_n=top_k * 10)

    rows = []
    for hit in raw:
        meta = hit['metadata']
        if gender and meta.get('gender') != gender:
            continue
        rating = float(meta.get('rating') or 0)
        if rating < min_rating:
            continue
        sim = float(hit['distance'])  # vectordb2 noemt het 'distance' maar het is similarity (0-1, hoger=beter)
        score = (1 - rating_weight) * sim + rating_weight * (rating / 5.0)
        rows.append({**meta, 'similarity': round(sim, 3), 'score': round(score, 3)})

    return (
        pd.DataFrame(rows)
        .sort_values('score', ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

In [ ]:
VIBE_HINTS = {
    'date':       'evening seductive sensual warm intimate',
    'office':     'clean professional subtle fresh daytime',
    'summer':     'fresh citrus light aquatic bright',
    'winter':     'warm amber spicy oud heavy cozy',
    'gym':        'fresh sporty clean light aquatic',
    'wedding':    'elegant sophisticated floral refined',
    'spicy':      'cinnamon pepper saffron cardamom clove ginger warm oriental',
    'sweet':      'vanilla caramel honey tonka gourmand',
    'woody':      'cedar sandalwood oud patchouli vetiver',
    'fresh':      'citrus bergamot lemon aquatic green',
    'floral':     'rose jasmine lily peony iris',
}

def expand_query(q: str) -> str:
    extras = [hint for keyword, hint in VIBE_HINTS.items() if keyword in q.lower()]
    return q if not extras else f"{q}. Related notes and vibe: {' '.join(extras)}"


def _get_score(hit: dict) -> float:
    """vectordb2 versions verschillen: 'distance', 'score', of 'similarity'."""
    for key in ('distance', 'score', 'similarity'):
        if key in hit:
            return float(hit[key])
    return 0.0


# Eenmalige probe — laat zien welke keys vectordb2 in deze versie teruggeeft
_probe = memory.search('test', top_n=1)
print('vectordb2 hit keys:', list(_probe[0].keys()) if _probe else '(geen resultaten)')


def recommend(query: str, top_k: int = 5, gender: str | None = None,
              min_rating: float = 0.0, rating_weight: float = 0.2) -> pd.DataFrame:
    """
    query         : vrije tekst, bijv. 'spicy perfume for a date'
    gender        : 'men' | 'women' | 'unisex' | None
    min_rating    : hard filter op Fragrantica-rating
    rating_weight : 0 = puur similarity, 1 = puur rating. 0.2 is meestal lekker.
    """
    expanded = expand_query(query)
    raw = memory.search(expanded, top_n=top_k * 10)

    rows = []
    for hit in raw:
        meta = hit.get('metadata', {}) or {}
        if gender and meta.get('gender') != gender:
            continue
        rating = float(meta.get('rating') or 0)
        if rating < min_rating:
            continue
        sim = _get_score(hit)
        score = (1 - rating_weight) * sim + rating_weight * (rating / 5.0)
        rows.append({**meta, 'similarity': round(sim, 3), 'score': round(score, 3)})

    return (
        pd.DataFrame(rows)
        .sort_values('score', ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

In [ ]:
recommend('spicy perfume for a date', top_k=5, min_rating=3.5)

In [ ]:
recommend('fresh summer office scent, not too strong', top_k=5, gender='men', min_rating=3.5)

In [ ]:
recommend('elegant floral for a wedding', top_k=5, gender='women', min_rating=4.0)

## Volgende stappen

- **Betere embedding_text**: `reviews_text` toevoegen (sentiment en echte gebruikerstaal) — als die kolom beschikbaar is in de dataset is dat waarschijnlijk de grootste kwaliteitssprong die we kunnen maken.
- **Hybride retrieval**: vector search vindt parfums op vibe-match, maar als de gebruiker een specifieke note noemt ("oud") wil je dat hard matchen. Combineren met een BM25/keyword pass zou dat oplossen.
- **LLM-laag**: top-k opsturen naar Claude of OpenAI om de aanbeveling te formuleren in natuurlijke taal — zelfde patroon als in `fragrantica_chatbot.ipynb`.
- **Vergelijken met ChromaDB**: dezelfde queries door beide stores halen en kijken welke beter ranked, P@5 zoals al gemeten in `fragrantica_chatbot.ipynb`.